# DAISY-BB-004: per-taxon crops and preprocessors (replayable)

Specimen crops labelled with GBIF order/family, one classification view per run (ORDER, FAMILY_Hymenoptera,
FAMILY_Diptera). Splits follow the source photo, so one photo never spans splits (leakage gate). A preprocessor is
USEFUL if test macro-F1 beats the majority-class macro-F1 by more than the frozen margin (0.10).

Four parts: (1) the frozen protocol and amendments resolve to their bytes, (2) the frozen crop report, with thin
classes flagged, (3) a crop build replayed on a small synthetic set (swap in the real drive), (4) results + custody
check for a run directory. **Swap data:** `DATASETS`, `REPORT`, `RUN_DIR`.

In [1]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == 'notebooks': os.chdir('..')
sys.path.insert(0, os.getcwd())
import hashlib, tempfile
import pandas as pd
from baebugs import taxon_crops as tc
from baebugs.explore import merkle_root

DATASETS = '/Volumes/magicBLACKbox/baebugs/datasets'           # swap: real datasets root (else a synthetic one is made)
REPORT = 'daisy/taxon_crops_report.json'                       # swap: a crop report
RUN_DIR = '/Volumes/magicBLACKbox/baebugs/daisy/local/004_v4'  # swap: a run dir with results.json + events.jsonl (+ weights)
for alt in ('daisy/results/DAISY-BB-004/v4',):                 # ingested copy in the repo, once it exists
    if not Path(RUN_DIR, 'results.json').exists() and Path(alt, 'results.json').exists(): RUN_DIR = alt
P = json.loads(Path('daisy/DAISY-BB-004.json').read_text())
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

## 1. Frozen protocol: the prereg hash resolves; amendments are added, never edited

In [2]:
pre = P['preregistration']
print(pre['path'], 'resolves' if sha(pre['path']) == pre['sha256'] else 'BREAK: prereg bytes changed')
pd.DataFrame([{'file': str(f), 'sha256': sha(f)[:16], 'last commit': os.popen(f'git log -1 --format=%h -- {f}').read().strip()}
              for f in sorted(Path('daisy').glob('DAISY-BB-004-prereg*.md'))])

daisy/DAISY-BB-004-prereg.md resolves


,file,sha256,last commit
0,daisy/DAISY-BB-004-prereg-amendment-1.md,58d7a9c79cc639ce,0fa0560
1,daisy/DAISY-BB-004-prereg-amendment-2.md,2c99e44bb68834d4,5f73d3a
2,daisy/DAISY-BB-004-prereg-amendment-3.md,065bbaef95d6c5c5,02ad4f8
3,daisy/DAISY-BB-004-prereg-amendment-4.md,cd1a03ad5a72f91b,fe2559c
4,daisy/DAISY-BB-004-prereg.md,5bf758d73c657d54,84a7e28


## 2. The frozen crop report, with thin classes flagged
A class with fewer than `THIN` test crops gives an unreliable F1; a run with no test split cannot be scored at all.
Both are reported, not hidden.

In [3]:
THIN = 5
rep = json.loads(Path(REPORT).read_text())
print(rep['crops'], 'crops', rep['by_split'], '| leakage media ids', rep['leakage_media_ids'], '| root', rep['merkle_root'][:16])
rows = []
for run, v in rep['runs'].items():
    for c in v['classes']:
        n = {s: v['counts'].get(s, {}).get(c, 0) for s in ('train', 'val', 'test')}
        flag = 'no test split' if not v['counts'].get('test') else 'no test crops' if n['test'] == 0 else f'thin (<{THIN})' if n['test'] < THIN else ''
        rows.append({'run': run, 'class': c, **n, 'flag': flag})
df = pd.DataFrame(rows); df[df.flag != '']

2627 crops {'train': 2015, 'val': 381, 'test': 231} | leakage media ids 0 | root 4d5f2831ed5ba285


,run,class,train,val,test,flag
1,ORDER,Diptera,489,85,0,no test crops
2,ORDER,Hemiptera,133,24,2,thin (<5)
8,FAMILY_Hymenoptera,Formicidae,22,4,1,thin (<5)
11,FAMILY_Hymenoptera,Vespidae,30,4,3,thin (<5)
13,FAMILY_Diptera,Bombyliidae,278,48,0,no test split
14,FAMILY_Diptera,Syrphidae,81,13,0,no test split
15,FAMILY_Diptera,Tachinidae,44,8,0,no test split
16,FAMILY_Diptera,other,36,5,0,no test split


In [4]:
df.pivot_table(index='run', values=['train', 'val', 'test'], aggfunc='sum')

,test,train,val
run,,,
FAMILY_Diptera,0,439,74
FAMILY_Hymenoptera,86,825,169
ORDER,231,2015,381


## 3. Replay the crop build
On magicstudio this rebuilds the real crops from the drive. Anywhere else it makes a small synthetic set (coloured
rectangles with polygon labels and a GBIF-shaped manifest) and runs the same `taxon_crops.build`, twice, to show the
Merkle root is deterministic and the leakage gate holds.

In [5]:
import numpy as np, cv2
def synthetic(root, n=30, seed=0):
    rng = np.random.default_rng(seed); root = Path(root); man = []
    orders = [('Hymenoptera', 'Apidae'), ('Coleoptera', 'Carabidae'), ('Diptera', 'Syrphidae')]
    for ds, sp, k in [('r3_ca', 'train', n), ('r3_ca', 'val', n // 3), ('holdout_az', 'val', n // 3)]:
        (root / ds / 'images' / sp).mkdir(parents=True, exist_ok=True); (root / ds / 'labels' / sp).mkdir(parents=True, exist_ok=True)
        for i in range(k):
            stem = f'{ds}_{sp}_{i}'; o, f = orders[i % 3]
            img = np.full((96, 128, 3), 235, np.uint8); x0, y0 = rng.integers(5, 50), rng.integers(5, 40)
            cv2.rectangle(img, (int(x0), int(y0)), (int(x0) + 40, int(y0) + 30), [(40, 90, 200), (30, 30, 30), (20, 160, 220)][i % 3], -1)
            cv2.imwrite(str(root / ds / 'images' / sp / f'{stem}.jpg'), img)
            xs, ys = [x0, x0 + 40, x0 + 40, x0], [y0, y0, y0 + 30, y0 + 30]
            (root / ds / 'labels' / sp / f'{stem}.txt').write_text('0 ' + ' '.join(f'{x / 128:.4f} {y / 96:.4f}' for x, y in zip(xs, ys)) + '\n')
            man.append({'file': f'{stem}.jpg', 'order': o, 'family': f, 'scientificName': f'{f} sp.', 'license': 'CC0', 'gbifID': i})
    (root / 'manifest.jsonl').write_text(''.join(json.dumps(m) + '\n' for m in man))
    return root / 'manifest.jsonl'

real = Path(DATASETS, 'r3_ca').exists()
if real:
    out = Path(tempfile.mkdtemp(prefix='taxon_crops_'))
    r1 = tc.build(DATASETS, out); r2 = None
else:
    syn = Path(tempfile.mkdtemp(prefix='taxon_syn_')); tc.MANIFESTS = [synthetic(syn)]; tc.MIN_TRAIN = 5
    r1 = tc.build(syn, syn / 'out1'); r2 = tc.build(syn, syn / 'out2')
print('real drive' if real else 'SYNTHETIC', '|', r1['crops'], 'crops', r1['by_split'], '| leakage', r1['leakage_media_ids'], '| root', r1['merkle_root'][:16])
if r2: print('rebuild root equal:', r1['merkle_root'] == r2['merkle_root'])
if real: print('matches frozen report:', r1['merkle_root'] == rep['merkle_root'])
{k: v['classes'] for k, v in r1['runs'].items()}

SYNTHETIC | 50 crops {'train': 30, 'val': 10, 'test': 10} | leakage 0 | root 4d632fe219b950db
rebuild root equal: True


{'ORDER': ['Coleoptera', 'Diptera', 'Hymenoptera'],
 'FAMILY_Hymenoptera': ['Apidae'],
 'FAMILY_Coleoptera': ['Carabidae']}

## 4. Results and custody for a run
Reads `results.json` + `events.jsonl` from `RUN_DIR`. Re-derives each verdict from the frozen margin, flags per-class
F1 of 0 and missing test splits, re-chains the event log, and checks every weights hash resolves to a stored file
(else BREAK). If no run directory is reachable here, it says so and stops: it never invents numbers.

In [6]:
res_p = Path(RUN_DIR, 'results.json')
if not res_p.exists():
    print('no results at', RUN_DIR, '-> run on magicstudio, or swap RUN_DIR to daisy/results/DAISY-BB-004/v4 once ingested')
else:
    R = json.loads(res_p.read_text()); margin = P['decision']['margin']
    print(R['protocol'], '| protocol sha ok:', R['protocol_sha256'] == sha('daisy/DAISY-BB-004.json'), '| custody', R.get('custody_status'))
    rows = []
    for name, r in R['results'].items():
        t = (r.get('eval') or {}).get('test')
        v = None if not t else ('USEFUL' if t['macro_f1'] - t['majority_macro_f1'] > margin else 'NULL')
        zero = [c for c, f in (t or {}).get('per_class_f1', {}).items() if f == 0]
        rows.append({'run': name, 'status': r['status'], 'test_n': (t or {}).get('n'), 'macro_f1': (t or {}).get('macro_f1'),
                     'majority': (t or {}).get('majority_macro_f1'), 'verdict': v, 'recorded': r.get('verdict'),
                     'agrees': v == r.get('verdict'), 'f1=0 classes': zero, 'note': '' if t else 'no test split: cannot score'})
    display(pd.DataFrame(rows))

no results at /Volumes/magicBLACKbox/baebugs/daisy/local/004_v4 -> run on magicstudio, or swap RUN_DIR to daisy/results/DAISY-BB-004/v4 once ingested


In [7]:
if res_p.exists():
    ev = [json.loads(l) for l in Path(RUN_DIR, 'events.jsonl').read_text().splitlines() if l.strip()]
    H = lambda o: hashlib.sha256(json.dumps(o, sort_keys=True, separators=(',', ':'), default=str).encode()).hexdigest()
    chain = all(e['leaf'] == H({k: v for k, v in e.items() if k != 'leaf'}) and e['prev'] == (ev[i - 1]['leaf'] if i else None) for i, e in enumerate(ev))
    print(len(ev), 'events | chain ok:', chain, '| events root ok:', merkle_root([e['leaf'] for e in ev]) == R['events_root'])
    for name, r in R['results'].items():
        w = r.get('weights_sha256'); f = Path(RUN_DIR, r['weights_file']) if r.get('weights_file') else None
        got = sha(f) if f and f.exists() else None
        print(f"{name:20s}", 'no weights (run failed)' if not w else 'RESOLVES' if got == w else f'BREAK: weights {w[:12]} not stored here')